# **Tahap 0: Cofiguration**

In [1]:
# Install dulu library buat baca RSS (Biasanya udah ada di Kaggle, tapi buat jaga-jaga)
%pip install -q feedparser pandas

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 2.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import feedparser
import pandas as pd
import urllib.parse
import re
from datetime import datetime
import torch
from transformers import pipeline

import os
import numpy as np
import random
import transformers as tf

from sklearn.preprocessing import MaxAbsScaler

In [3]:
# Maksa GPU (CuDNN) di Kaggle biar ga ngacak hitungan
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['PYTHONHASHSEED'] = str(42)

def kunci_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

kunci_seed(42)

In [4]:
MARKET_POSITIVE_WORDS = [
    "naik", "menguat", "positif", "tumbuh", "meningkat", "optimis", "rebound", 
    "bullish", "stabil", "surplus", "rekor", "cuan", "laba", "akumulasi", "prospek"
]

MARKET_NEGATIVE_WORDS = [
    "turun", "melemah", "negatif", "anjlok", "koreksi", "tertekan", "inflasi", 
    "resesi", "krisis", "rugi", "bearish", "risiko", "ketidakpastian", "perang", 
    "konflik", "gagal", "defisit"
]

STOPWORDS_ID = {
    "dan", "yang", "di", "ke", "dari", "untuk", "dengan", "dalam", "hari", 
    "ini", "terbaru", "adalah", "pada", "karena", "sebagai", "itu", "akan", 
    "bisa", "ada", "tidak", "juga", "sudah", "saja", "lagi", "atau", "oleh",
    "untuk", "kita", "kami", "saya", "kamu", "mereka", "dia", "saat", "bagi"
}

# **TAHAP 1 NLP: Transplantasi Otak Sentimen.**

* **Mempertahankan Scraper Emas:** Kita tetep pakai `feedparser` dan `urllib.parse` dari `app.py` lu karena itu udah standar industri buat nyedot RSS Google News secara bersih dan terstruktur.
* **Injeksi HuggingFace Transformers:** Kita bakal manggil *library* `transformers` buat nge-*load* model `mdhugol/indonesia-bert-sentiment-classifier`. Karena lu lagi duduk manis di Kaggle dengan fasilitas Dual T4 GPU (VRAM 15GB), kita bakal nyuruh model ini langsung jalan di GPU (`device=0`) biar eksekusinya kilat.
* **Transisi ke Softmax Probability:** AI lu nggak akan lagi ngeluarin teks kaku "Positif" atau "Negatif". Model ini akan membaca kalimat utuh dan mengeluarkan probabilitas desimal dari *layer Softmax*. Contoh: *Headline* berita A memiliki probabilitas 80% Positif, 15% Netral, dan 5% Negatif. Angka-angka halus inilah yang sangat dibutuhkan oleh sistem *Fuzzy Logic* lu nanti.

In [5]:
NEWS_KEYWORDS = {
    "ANTM": ["ANTM saham", "Antam emas", "Aneka Tambang saham", "ANTM emas"],
    "MDKA": ["MDKA saham", "Merdeka Copper Gold saham", "MDKA emas"],
    "BRMS": ["BRMS saham", "Bumi Resources Minerals saham", "BRMS emas"],
    "PSAB": ["PSAB saham", "J Resources Asia Pasifik saham", "PSAB emas", "J Resources gold", "J Resources tambang emas"],
    "ARCI": ["Archi Indonesia", "Archi", "ARCI", "PT Archi", "Tambang Archi"],
    "UNTR": ["United Tractors", "United Tractor", "UNTR", "PT United Tractors", "UT"],
    "AMMN": ["Amman Mineral", "Amman", "AMMN", "PT Amman Mineral", "Amman Internasional"],
    "HRTA": ["Hartadinata", "Hartadinata Abadi", "HRTA", "PT Hartadinata", "Harta Abadi",],
}

MARKET_KEYWORDS =  ["harga emas hari ini", "emas dunia", "emas antam", "harga emas naik", "harga emas turun", "gold price today", 
                    "IHSG hari ini", "saham Indonesia", "pasar modal Indonesia", "Bursa Efek Indonesia", "rekomendasi saham hari ini",
                    "ekonomi Indonesia", "inflasi Indonesia", "suku bunga Bank Indonesia", "rupiah hari ini", "pertumbuhan ekonomi Indonesia",
                    "ekonomi global", "The Fed suku bunga", "inflasi Amerika", "resesi global", "geopolitical risk market",
                    "harga komoditas", "harga minyak dunia", "harga batu bara", "harga tembaga", "commodity market",
                    "saham ramai dibahas", "market trend today", "investor sentiment", "fear of missing out saham", "berita ekonomi viral"
]

In [6]:
# ======================================================
# 1. LOAD MODEL INDOBERT (OTAK BARU)
# ======================================================
# from transformers import AutoTokenizer, AutoModelForSequenceClassification
otak_nlp = pipeline(
    "text-classification", 
    model="mdhugol/indonesia-bert-sentiment-classification", 
    # tokenizer = AutoTokenizer.from_pretrained("mdhugol/indonesia-bert-sentiment-classification"),
    top_k=None, # Biar semua probabilitas (Pos, Neu, Neg) keluar
    device=0 
)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: mdhugol/indonesia-bert-sentiment-classification
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [7]:
def bersihkan_teks(teks):
    # 1. Lowercase: IndoBERT versi ini sensitif sama huruf kapital
    teks = teks.lower()
    
    # 2. Hapus URL/Link (kalau tiba-tiba ada nyelip di RSS)
    teks = re.sub(r'http\S+|www\S+|https\S+', '', teks, flags=re.MULTILINE)
    
    # 3. Hapus entitas Twitter (Mention & Hashtag) karena modelnya dari data sosmed
    teks = re.sub(r'\@\w+|\#\w+', '', teks)
    
    # 4. Hapus karakter HTML (kayak &amp;, &quot;)
    teks = re.sub(r'&[a-z]+;', ' ', teks)
    
    # 5. Hapus tanda baca berlebihan dan karakter non-alfanumerik 
    # (Kita sisain angka karena buat saham angka 20% itu penting)
    teks = re.sub(r'[^\w\s]', ' ', teks)
    
    # 6. Hapus spasi ganda dari hasil pemotongan di atas
    teks = re.sub(r'\s+', ' ', teks).strip()
    
    return teks

In [8]:
def sedot_berita_expert(ticker, keywords_list, max_per_keyword=10):
    print(f"[*] (Scraping & Preprocessing) Memproses: {ticker}...")
    news_items = []
    
    for keyword in keywords_list:
        query = urllib.parse.quote(f"{keyword} when:30d")
        url = f"https://news.google.com/rss/search?q={query}&hl=id&gl=ID&ceid=ID:id"
        feed = feedparser.parse(url)
        
        for entry in feed.entries[:max_per_keyword]:
            judul_kotor = entry.title if "title" in entry else ""
            judul_bersih_portal = judul_kotor.rsplit(" - ", 1)[0] if " - " in judul_kotor else judul_kotor

            judul_siap_nlp = bersihkan_teks(judul_bersih_portal)
            
            # Lewatin kalau judulnya kosong setelah dibersihin
            if not judul_siap_nlp: 
                continue
                
            # label_index = {'LABEL_0': 'positive', 'LABEL_1': 'neutral', 'LABEL_2': 'negative'}
            # status = label_index[hasil_bert['label']]
            # score = hasil_bert['score']
            hasil_bert = otak_nlp(judul_siap_nlp)[0]
            prob_pos, prob_neu, prob_neg = 0.0, 0.0, 0.0
            
            for sentimen in hasil_bert:
                if sentimen['label'] == 'LABEL_0': prob_pos = sentimen['score']
                elif sentimen['label'] == 'LABEL_1': prob_neu = sentimen['score']
                elif sentimen['label'] == 'LABEL_2': prob_neg = sentimen['score']
                    
            news_items.append({
                "Ticker": ticker,
                "Keyword": keyword,
                "Tanggal": entry.published if "published" in entry else None,
                "Judul_Asli": judul_bersih_portal,
                "Judul_Preprocessed": judul_siap_nlp, # Biar lu bisa inspeksi bedanya
                "Prob_Positif": prob_pos,
                "Prob_Netral": prob_neu,
                "Prob_Negatif": prob_neg
            })
            # news_items.append({
            #     "Ticker": ticker,
            #     "Keyword": keyword,
            #     "Tanggal": entry.published if "published" in entry else None,
            #     "Judul_Asli": judul_bersih_portal,
            #     "Judul_Preprocessed": judul_siap_nlp, # Biar lu bisa inspeksi bedanya
            #     "kelas": status,
            #     "confidence": score
            # })
            
    df_news = pd.DataFrame(news_items)
    
    if not df_news.empty:
        df_news["Tanggal"] = pd.to_datetime(df_news["Tanggal"], errors="coerce").dt.date
        df_news = df_news.dropna(subset=["Judul_Asli"]).drop_duplicates(subset=["Judul_Asli"])
        
    return df_news

In [9]:
# ======================================================
# 3. EKSEKUSI MASSAL (Semua Emiten)
# ======================================================
lemari_sentimen = {}

for ticker, list_keyword in NEWS_KEYWORDS.items():
    lemari_sentimen[ticker] = sedot_berita_expert(ticker, list_keyword, max_per_keyword=15)

[*] (Scraping & Preprocessing) Memproses: ANTM...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


[*] (Scraping & Preprocessing) Memproses: MDKA...
[*] (Scraping & Preprocessing) Memproses: BRMS...
[*] (Scraping & Preprocessing) Memproses: PSAB...
[*] (Scraping & Preprocessing) Memproses: ARCI...
[*] (Scraping & Preprocessing) Memproses: UNTR...
[*] (Scraping & Preprocessing) Memproses: AMMN...
[*] (Scraping & Preprocessing) Memproses: HRTA...


In [10]:
# Cek hasil buat salah satu saham, misal PSAB
pd.set_option('display.max_colwidth', None)
print("\n📊 CONTOH HASIL PANEN BERITA (PSAB):")
lemari_sentimen['PSAB'][['Keyword', 'Judul_Asli', 'Judul_Preprocessed', 'Prob_Positif', 'Prob_Negatif']].head(50)
# lemari_sentimen['PSAB'][['Keyword', 'Judul_Asli', 'Judul_Preprocessed', 'kelas', 'confidence']].head(50)


📊 CONTOH HASIL PANEN BERITA (PSAB):


,Keyword,Judul_Asli,Judul_Preprocessed,Prob_Positif,Prob_Negatif
0,PSAB saham,"PSAB tebar dividen Rp2,77 triliun, yield setara 22,3%",psab tebar dividen rp2 77 triliun yield setara 22 3,0.003729,0.000610
1,PSAB saham,"Rekomendasi Saham Hari Ini: AMMN, BMRI, PGAS & PSAB, IHSG Diprediksi Rebound Terbatas",rekomendasi saham hari ini ammn bmri pgas psab ihsg diprediksi rebound terbatas,0.001965,0.000612
2,PSAB saham,"Tutup 2025, Laba PSAB Melompat 285 Persen",tutup 2025 laba psab melompat 285 persen,0.001296,0.002131
3,PSAB saham,"Dividen Jumbo PSAB Disetujui, Pemegang Saham Dapat Rp105 per Lembar",dividen jumbo psab disetujui pemegang saham dapat rp105 per lembar,0.002282,0.000492
4,PSAB saham,"Dividen PSAB Rp105 per Saham Distribusi 30 Juni, Cek Jadwal Lengkapnya",dividen psab rp105 per saham distribusi 30 juni cek jadwal lengkapnya,0.001566,0.000497
5,PSAB saham,FAC Sekuritas,fac sekuritas,0.083338,0.016132
6,PSAB saham,"Laba Melonjak Lebih dari 2.000 Persen, Saham PSAB Kembali Jadi Sorotan Investor",laba melonjak lebih dari 2 000 persen saham psab kembali jadi sorotan investor,0.002145,0.001311
7,PSAB saham,"PSAB Tebar Dividen Rp2,7 Triliun, Rasio Pembayaran Tembus 448 Persen",psab tebar dividen rp2 7 triliun rasio pembayaran tembus 448 persen,0.004650,0.000487
8,PSAB saham,"IHSG Cenderung Lanjutkan Pelemahan, Adapun Saham EMAS dan PGAS Layak Dicermati",ihsg cenderung lanjutkan pelemahan adapun saham emas dan pgas layak dicermati,0.008875,0.000813
9,PSAB saham,"Emiten Emas PSAB Tebar Dividen Rp2,78 Triliun, Catat Jadwal Selengkapnya",emiten emas psab tebar dividen rp2 78 triliun catat jadwal selengkapnya,0.002701,0.000488


# **2. Agregasi Harian & Kalibrasi Sentimen.**

* **Grouping Temporal (Penyelarasan Dimensi):** Model XGBoost lu yang kemaren itu murni nerima data harian (1 baris per hari). Kalau hari ini ada 15 berita tentang BRMS yang rilis, lu nggak mungkin masukin 15 baris ini ke sistem Fusi. Kita harus melebur (agregasi) 15 probabilitas tersebut menjadi 1 nilai rata-rata (*mean*) representatif untuk hari itu.
* **Koreksi Normalisasi (Softmax Calibration):** *Output* probabilitas mentah dari *pipeline* HuggingFace kadang terdistorsi kalau teksnya kurang panjang. Kita harus memaksa total probabilitas (Positif + Netral + Negatif) menjadi mutlak 1.0 (100%) secara matematis agar tidak ada bias distribusi.
* **Reduksi Dimensi (Skor Tunggal):** Sistem *Fuzzy Inference Mamdani* lu bakal mabok kalau dikasih 3 variabel probabilitas cuma buat merepresentasikan 1 sentimen. Kita harus kompres angka ini pakai rumus `(Prob_Positif - Prob_Negatif)`, sehingga ngasilin satu parameter tunggal dengan rentang -1 (Bearish/Negatif Parah) sampai +1 (Bullish/Positif Parah).

In [11]:
def agregasi_sentimen_harian(df_news):
    print(f"[*] Melakukan kompresi dan agregasi harian...")
    
    if df_news.empty:
        return pd.DataFrame()

    # 1. GROUPING BERDASARKAN TANGGAL
    # Mengambil rata-rata dari seluruh probabilitas berita di hari perdagangan yang sama
    df_harian = df_news.groupby(['Ticker', 'Tanggal']).agg(
        Total_Berita=('Judul_Asli', 'count'), # Ngitung berapa berita yang nge-backup sentimen hari ini
        Avg_Prob_Positif=('Prob_Positif', 'mean'),
        Avg_Prob_Netral=('Prob_Netral', 'mean'),
        Avg_Prob_Negatif=('Prob_Negatif', 'mean')
    ).reset_index()
    
    # 2. KOREKSI NORMALISASI ABSOLUT
    # Memastikan total probabilitas = 1.0 (100%) buat menghilangkan distorsi Softmax
    total_prob = df_harian['Avg_Prob_Positif'] + df_harian['Avg_Prob_Netral'] + df_harian['Avg_Prob_Negatif']
    
    # Pake np.where buat ngehindarin error division by zero (jaga-jaga kalau probnya 0 semua)
    df_harian['Avg_Prob_Positif'] = np.where(total_prob > 0, df_harian['Avg_Prob_Positif'] / total_prob, 0)
    df_harian['Avg_Prob_Netral'] = np.where(total_prob > 0, df_harian['Avg_Prob_Netral'] / total_prob, 0)
    df_harian['Avg_Prob_Negatif'] = np.where(total_prob > 0, df_harian['Avg_Prob_Negatif'] / total_prob, 0)
    
    # 3. PENCIPTAAN SKOR TUNGGAL (Amunisi buat Fuzzy Logic)
    # Rentang: -1 (Sangat Negatif) sampai +1 (Sangat Positif)
    df_harian['Skor_Sentimen_Final'] = df_harian['Avg_Prob_Positif'] - df_harian['Avg_Prob_Negatif']
    
    # Urutin dari tanggal terbaru ke terlama biar bacanya enak
    df_harian = df_harian.sort_values(by='Tanggal', ascending=False).reset_index(drop=True)
    
    return df_harian

In [12]:
# ======================================================
# EKSEKUSI AGREGASI BUAT SEMUA SAHAM
# ======================================================
lemari_sentimen_harian = {}

scaler_sentimen = MaxAbsScaler()

for ticker, df_mentah in lemari_sentimen.items():
    lemari_sentimen_harian[ticker] = agregasi_sentimen_harian(df_mentah)
    skor_mentah = lemari_sentimen_harian[ticker]['Skor_Sentimen_Final'].values.reshape(-1, 1)
    lemari_sentimen_harian[ticker]['Skor_Sentimen_Final'] = scaler_sentimen.fit_transform(skor_mentah)

[*] Melakukan kompresi dan agregasi harian...
[*] Melakukan kompresi dan agregasi harian...
[*] Melakukan kompresi dan agregasi harian...
[*] Melakukan kompresi dan agregasi harian...
[*] Melakukan kompresi dan agregasi harian...
[*] Melakukan kompresi dan agregasi harian...
[*] Melakukan kompresi dan agregasi harian...
[*] Melakukan kompresi dan agregasi harian...


In [13]:
# Kita inspeksi hasilnya buat PSAB
print("\n📊 HASIL AGREGASI HARIAN (PSAB):")
print(lemari_sentimen_harian['PSAB'])


📊 HASIL AGREGASI HARIAN (PSAB):
   Ticker     Tanggal  Total_Berita  Avg_Prob_Positif  Avg_Prob_Netral  \
0    PSAB  2026-06-07             1          0.026466         0.972483   
1    PSAB  2026-06-06             2          0.001891         0.997241   
2    PSAB  2026-06-05            10          0.017672         0.980208   
3    PSAB  2026-06-04             1          0.001484         0.997695   
4    PSAB  2026-06-03             6          0.001826         0.997210   
5    PSAB  2026-06-02             6          0.004582         0.994400   
6    PSAB  2026-05-29             1          0.001106         0.997680   
7    PSAB  2026-05-28             1          0.008875         0.990312   
8    PSAB  2026-05-26             1          0.001744         0.996765   
9    PSAB  2026-05-24             1          0.002836         0.995054   
10   PSAB  2026-05-22             4          0.001729         0.997220   
11   PSAB  2026-05-19             1          0.001965         0.997422   
12   

# **Tahap 3: Export Output NLP (.csv/.json)**
* **Fakta Teknis:** Sama kayak model deret waktu kemarin, hasil olahan skor sentimen dari IndoBERT ini harus lu ekspor jadi file dataset terpisah (misal `brms_sentiment_output.csv`). File inilah yang nantinya bakal dijajalin berdampingan dengan prediksi XGBoost di depan meja persidangan Fuzzy Logic Mamdani (Tahap Akhir).

In [14]:
# Bikin folder khusus biar rapi
os.makedirs('data_sentimen', exist_ok=True)

for ticker, df in lemari_sentimen_harian.items():
    if not df.empty:
        nama_file = f"data_sentimen/sentimen_{ticker}.csv"
        df.to_csv(nama_file, index=False)
        print(f"   [+] Tersimpan: {nama_file}")

   [+] Tersimpan: data_sentimen/sentimen_ANTM.csv
   [+] Tersimpan: data_sentimen/sentimen_MDKA.csv
   [+] Tersimpan: data_sentimen/sentimen_BRMS.csv
   [+] Tersimpan: data_sentimen/sentimen_PSAB.csv
   [+] Tersimpan: data_sentimen/sentimen_ARCI.csv
   [+] Tersimpan: data_sentimen/sentimen_UNTR.csv
   [+] Tersimpan: data_sentimen/sentimen_AMMN.csv
   [+] Tersimpan: data_sentimen/sentimen_HRTA.csv


In [15]:
import os
import shutil

models_dir = "/kaggle/working/data_sentimen"
output_zip = "/kaggle/working/data_sentimen"

if not os.path.isdir(models_dir):
    print(f"Directory not found: {models_dir}")
else:
    # remove existing zip if any
    if os.path.exists(output_zip + ".zip"):
        os.remove(output_zip + ".zip")
    shutil.make_archive(output_zip, 'zip', models_dir)
    zip_path = output_zip + ".zip"
    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"Created: {zip_path} ({size_mb:.2f} MB)")

Created: /kaggle/working/data_sentimen.zip (0.01 MB)
